# Domain-adaptive pretraining — MLM trên Kanglish

**Settings:** Accelerator = **GPU T4 ×2** · Internet = **On** · ~25 phút

Từ vựng của MuRIL được xây cho tiếng Ấn viết bằng **chữ bản địa**, nên Kanglish chữ Latin bị xé
vụn: **2,05 mảnh/từ** so với **1,04** của tiếng Anh, và chỉ **36,7%** từ còn nguyên một mảnh.

```
government    ->  ['government']                        (tiếng Anh, 1 mảnh)
Bajetigu      ->  ['Ba', '##jet', '##ig', '##u']        (Kanglish, 4 mảnh)
```

Embedding của `##jet`, `##ikk` được học trong ngữ cảnh **chẳng liên quan gì** tới tiếng Kannada.
MLM kéo chúng về đúng chỗ — và vì nó **không cần nhãn**, nó tránh được đúng vấn đề đã giết chết
phương án nối dữ liệu ngoài vào train (đo được +0,0012 ± 0,0087, tức bằng không): *"offensive"*
khác *"hate"*, nhưng văn bản thì vẫn cùng một ngôn ngữ.

**Nói thẳng về quy mô:** kho của bạn ~**0,31 triệu token**, trong khi MuRIL pretrain trên ~16 **tỷ**
và các bài DAPT thường dùng 1–100 triệu. Thứ cứu vớt là chỉ **8.405 mục từ vựng (4,3%)** thực sự
xuất hiện, nên toàn bộ ngân sách dồn đúng vào những embedding đang sai. Kỳ vọng **+0,01 đến
+0,03**, không phải bước nhảy.

In [ ]:
import os, subprocess, sys

REPO, BRANCH, WORK = "trong5nhan6/Text", "main", "/kaggle/working"
TOKEN = ""
try:
    from kaggle_secrets import UserSecretsClient
    TOKEN = UserSecretsClient().get_secret("GH_TOKEN")
except Exception:
    pass
url = f"https://{TOKEN + '@' if TOKEN else ''}github.com/{REPO}.git"
hide = (lambda s: s.replace(TOKEN, "***")) if TOKEN else (lambda s: s)

os.chdir(WORK)
cmd = (["git", "-C", "repo", "pull", "--ff-only"] if os.path.isdir("repo/.git")
       else ["git", "clone", "--depth", "1", "-b", BRANCH, url, "repo"])
r = subprocess.run(cmd, capture_output=True, text=True)
print(hide((r.stdout + r.stderr).strip()))
if r.returncode:
    raise SystemExit("git that bai -- kiem tra Internet = On, repo Public")

os.chdir(f"{WORK}/repo"); sys.path.insert(0, os.getcwd())
print(subprocess.run(["git", "log", "--oneline", "-1"], capture_output=True, text=True).stdout.strip())

In [ ]:
!pip -q install ftfy sentencepiece
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

## 1) Kho văn bản

Gom **mọi** file có Kanglish, khử trùng lặp. Không cần nhãn nào.

> **Lát held-out bị loại khỏi kho, mặc định.** Văn bản đó không mang nhãn nên giữ lại cũng không
> rò rỉ nhãn — nhưng model sẽ đã đọc đúng những câu ấy, và mọi macro-F1 đo trên lát đó sau này
> sẽ **lạc quan một cách âm thầm**. Giữ trung thực chỉ tốn 919 dòng (kho còn khoảng 13,5 nghìn dòng).
>
> Ngược lại, `*_validation_inputs.csv` và file test (`hastika_*_test.csv`) **được** đưa vào: đó là transductive learning
> thông thường, và đúng là tình huống bản nộp sẽ chạy. Nhớ khai báo điều này trong bài báo.

In [ ]:
from pretrain_mlm import build_corpus
from src.utils.config import load_config
from src.data.preprocessing import ensure_processed

cfg = load_config("configs/base.yaml", task="a")
ensure_processed(cfg)
corpus = build_corpus(cfg, include_eval=False)
print(f"-> {len(corpus):,} dong")
for t in corpus[:5]:
    print("   ", t[:80])

## 2) Chạy MLM — bản 2

So với bản 1 (`checkpoints/mlm/muril-base-cased`):

| | bản 1 | **bản 2** |
|---|---|---|
| Cách che | từng mảnh (`th [MASK]` → nhìn `th` là đoán ra) | **cả từ** `--wwm` (`[MASK] [MASK]` → phải đọc câu) |
| Kho | train + val | **+ file test** (`hastika_*_test.csv`, không dùng nhãn) |
| Xuất phát | `google/muril` | **`cnerg_muril`** (+ bản `google/muril` để so) |
| Đầu MLM | có sẵn | cnerg **không có** → chép từ `google/muril` (`--mlm_head_from`) |
| Khi nào dừng | lưu epoch cuối | tách **5%** làm tập đánh giá (`--eval_ratio`), **lưu epoch tốt nhất** |

**Đọc log:** mỗi epoch in `loss/ppl` (trên kho train) và `eval loss/ppl` (trên 5% tách riêng).
`*` = epoch tốt nhất đến lúc đó, và checkpoint được lưu ngay lúc ấy. Nếu `eval ppl` bắt đầu
**tăng** trong khi `ppl` train vẫn giảm, model đang học thuộc kho.

> **Đừng hoảng với số đầu của cnerg.** Đo ở máy: trước khi train, `cnerg_muril` có eval loss
> **14,9**, còn tệ hơn đoán đều (12,2), vì các tầng trên của nó đã bị fine-tune cho phân loại nên
> đầu MLM chép sang không khớp. Nhưng chỉ sau 24 bước nó xuống **8,4**, dưới cả `google/muril` gốc
> (8,6 với cùng cách che cả từ). Với vài nghìn bước trên Kaggle nó hồi phục hẳn.
>
> Che cả từ **khó hơn** che từng mảnh (google/muril: loss 8,6 so với 7,8). Vì vậy ppl của bản 2
> **không so được** với ppl của bản 1. Chỉ so được bằng macro-F1 sau fine-tune, ở mục 3.

In [ ]:
# ============================ MLM ban 2 ============================
MLM_RUNS = [   # (checkpoint xuat phat, chep dau MLM tu, thu muc luu) -- moi dong ~30 phut tren 2xT4
    ('Hate-speech-CNERG/kannada-codemixed-abusive-MuRIL', 'google/muril-base-cased', 'checkpoints/mlm_v2/cnerg-muril'),
    ('google/muril-base-cased',                           None,                      'checkpoints/mlm_v2/muril'),
]
MLM_EPOCHS = 15     # tran; epoch tot nhat theo eval ppl moi duoc luu
WWM        = True   # che ca tu
EVAL_RATIO = 0.05   # 5% kho lam tap danh gia MLM (0 = tat, luu epoch cuoi nhu ban 1)
BATCH      = 32     # TONG, chia deu cho cac GPU (2 x T4 -> 16/GPU). Giam neu OOM.
GRAD_ACCUM = 1      # BATCH x GRAD_ACCUM = batch hieu dung
MAX_LEN    = 128    # token
LR         = 5e-5

import time
for model, head, out in MLM_RUNS:
    flags = (f"--model {model} --out {out} --epochs {MLM_EPOCHS} --batch_size {BATCH} "
             f"--grad_accum {GRAD_ACCUM} --max_len {MAX_LEN} --lr {LR} --eval_ratio {EVAL_RATIO}"
             + (" --wwm" if WWM else "") + (f" --mlm_head_from {head}" if head else ""))
    print("=" * 72, f"\n{model}  ->  {out}\n" + "=" * 72, flush=True)
    t0 = time.time()
    !python pretrain_mlm.py {flags}
    print(f"-> {(time.time() - t0) / 60:.1f} phut")

# THU NHANH truoc khi bo 1 gio GPU (chay het ca phan luu, ~1-2 phut):
# !python pretrain_mlm.py --model Hate-speech-CNERG/kannada-codemixed-abusive-MuRIL --mlm_head_from google/muril-base-cased --wwm --eval_ratio 0.2 --max_rows 200 --epochs 1 --out /tmp/mlm_test
# NEU OOM: ha BATCH va tang GRAD_ACCUM (vd 16 x 2). CHI 1 GPU: them --single_gpu vao flags.

> **Về OOM.** Logits của MLM có shape `[batch, len, vocab]`, mà vocab của MuRIL là
> **197.285**, nên riêng một tensor đó ở `batch 32 × len 128` đã chiếm **3,2 GB**, và phải giữ
> hai bản (xuôi + ngược). Đó là thứ làm nổ T4, không phải model.
>
> **Trên 2×T4 script tự dùng cả hai GPU** (`DataParallel`), nên `BATCH` là **tổng** và mỗi GPU
> chỉ giữ một nửa:
>
> | BATCH tổng | /GPU | logits/GPU | **GPU 0** | GPU 1 |
> |---|---|---|---|---|
> | 16 | 8 | 1,62 GB | 6,4 GB | 3,6 GB |
> | **32** | **16** | **3,23 GB** | **8,0 GB** | 5,2 GB |
> | 48 | 24 | 4,85 GB | 9,7 GB | 6,8 GB |
> | 64 | 32 | 6,46 GB | 11,3 GB | 8,4 GB |
>
> GPU 0 luôn chật hơn vì chỉ nó giữ trọng số gốc, gradient và hai trạng thái AdamW (3,8 GB).
> Che cả từ và tập đánh giá **không** tốn thêm VRAM. Script in ước lượng VRAM **trước khi** train.

## 3) Fine-tune từ checkpoint vừa thích nghi

Không cần code mới: chỉ trỏ `model.name` vào thư mục vừa lưu. Mỗi dòng trong `PAIRS` chạy **bản
gốc** và **bản MLM** cho cả hai task. Bản gốc đã chạy rồi thì tự bỏ qua. `run_name` tự gắn đuôi
theo thư mục (`_mlm`, `_mlm-v2`) nên các bản không bao giờ đè lên nhau.

In [ ]:
import os
PAIRS = [   # (config, checkpoint MLM tuong ung)
    ('cnerg_muril', 'checkpoints/mlm_v2/cnerg-muril'),   # MLM ban 2 tu cnerg
    ('muril',       'checkpoints/mlm_v2/muril'),         # MLM ban 2 tu google/muril
    ('muril',       'checkpoints/mlm/muril-base-cased'), # MLM ban 1 (neu co) -- de so ban 1 va ban 2
]
FT_EPOCHS = 6     # so epoch khi FINE-TUNE -- KHAC voi MLM_EPOCHS o tren
SUF = f'_e{FT_EPOCHS}'
# patience = epochs: chay tron lich LR roi giu epoch tot nhat
FT = f'--set training.epochs={FT_EPOCHS} training.early_stopping_patience={FT_EPOCHS}'

for cfg, ckpt in PAIRS:
    if not os.path.isfile(f'{ckpt}/config.json'):
        print(f"bo qua {ckpt}: chua co"); continue
    for t in ('a', 'b'):
        print("=" * 70, flush=True)
        !python train.py --config configs/{cfg}.yaml --task {t} {FT} --run_suffix {SUF}
        !python train.py --config configs/{cfg}.yaml --task {t} {FT} model.name={ckpt} --run_suffix {SUF}

In [ ]:
import pandas as pd
d = pd.read_csv('results/metrics.csv')
display(d[d.run.str.contains('muril') & d.run.str.endswith(SUF)][['task', 'run', 'macro_f1', 'accuracy', 'best_epoch']]
        .sort_values(['task', 'macro_f1'], ascending=[True, False]))

## 4) Tải checkpoint về

Mỗi checkpoint MLM khoảng **1,5 GB** (fp32, kèm đầu MLM). Nén **riêng từng cái** để tải từng file
và để dùng trong `embed_mix.ipynb` hoặc session Kaggle khác (upload thành Kaggle Dataset hoặc
Google Drive). Cấu trúc bên trong zip giữ nguyên đường dẫn, nên giải nén ở gốc repo là dùng được.

In [ ]:
%cd /kaggle/working/repo
import glob, os
for d in sorted(glob.glob('checkpoints/mlm_v2/*')):
    z = f"/kaggle/working/{d.replace('/', '_')}.zip"
    !zip -r -q {z} {d}
    print(f"{z}: {os.path.getsize(z) / 1e9:.2f} GB")